# Modelo RNN (LSTM) — falha de porta em 7 dias

O Modelo 5 de Random Forest testavaao  para ver se o histórico de alarmes de uma porta prevê uma falha nos próximos 7 dias, usando features feitas em janelas móveis (1/3/7/14/30 dias). Este notebook testa a mesma pergunta com uma rede neural recorrente (RNN), que recebe uma sequência não processada de alarmes e aprende sozinha o que importa olhar no tempo, sem ter alguém para desenhar as janelas na mão.

## 1. Comparação com o Modelo 5

Foi pensaod que para uma comparação entre dois modelos fazer sentido, os dois precisam responder à mesma pergunta com mesmo alvo e mesmas métricas. Assim a diferença pode ser atribuida às diferentes arquiteturas.

Este modelo usa a sequência de alarmes por porta específica, a mesma unidade de análise do Modelo 5 (porta-dia) e o mesmo alvo (`y = 1` se houve uma ordem corretiva emergencial, ZM01, nos 7 dias seguintes). Assim a comparação é feita com o Modelo 5, os dois preveem  a mesma coisa, só que com estruturas diferentes.

## 2. O que é uma Rede Neural Recorrente (RNN)

O Random Forest recebe uma linha por porta por dia com números já calculados; quantos alarmes teve nos últimos 7 dias, nos últimos 30, etc. Cada uma dessas colunas foi pensada antes de treinar o modelo.

Porém com uma RNN esse processo é diferente: ela recebe a sequência de dias, um de cada vez, na ordem em que aconteceram, e processa cada dia levando em conta o que ela "lembra" dos dias anteriores. Ao processar o dia 2, ela já sabe o que viu no dia 1; ao processar o dia 3, ela já sabe o que aconteceu nos dias 1 e 2, e assim por diante até o último dia da sequência (nesse caso, os 30 dias antes de `t0`). Esse resumo do que já viu até agora se chama hidden state, e é ele que carrega a memória de um dia pro outro.

No fim da sequência, o hidden state guarda um resumo dos 30 dias, aprendido pela própria rede durante o treino, e não desenhado à mão como as janelas móveis do Modelo 5. Uma última camada transforma esse resumo em uma probabilidade de falha.

## 3. Por que LSTM em vez de uma RNN simples

O problema de uma RNN simples é que, conforme a sequência cresce, a rede vai esquecendo o que viu muitos passos atrás, ou seja informação do dia 1 praticamente some do hidden state até chegar no dia 30.

**LSTM** (Long Short-Term Memory) resolve isso adicionando uma segunda memória, mais estável, que passa de um dia pro outro quase sem mudar, a não ser que a rede decida explicitamente atualizar ou apagar parte dela. Ela faz isso com três gates, que a própria rede aprende a usar durante o treino:

- **portão de esquecimento:** decide o que apagar da memória;
- **portão de entrada:** decide o que adicionar do dia atual;
- **portão de saída:** decide o que passar adiante pro próximo dia.

Essas decisões não são programadas, a rede aprende os pesos desses portões olhando para os dados de treino, do mesmo jeito que o Random Forest aprende onde cortar cada árvore olhando pros dados. Na prática, isso significa que a LSTM consegue notar, por exemplo, que uma porta teve um alarme importante há 3 semanas mesmo que os dias mais recentes estejam calmos, algo que uma RNN simples teria "esquecido".

## 4. Como as sequências foram montadas

As sequências foram montadas de forma que:

- No Modelo 5, cada observação era uma linha, representando uma porta em um dia específico.
- Na RNN, cada observação vira uma sequência dos últimos 30 dias daquela porta, em ordem cronológica.

Ou seja para uma porta específica, por exemplo `BUT_1_08`, e um dia de previsão `t0 = 15/06/2024`.

No Modelo 5, a linha daquela porta naquele dia seria algo como:

| porta | data | n_CA_7d | n_CA_30d | ... | y |
|---|---|---:|---:|---:|---:|
| BUT_1_08 | 15/06/2024 | 3 | 12 | ... | 1 |

Assim, ele recebe um resumo do histórico: “quantos alarmes aconteceram nos últimos 7 dias, 30 dias, etc.”

Já para a RNN, em vez de um resumo, a ideia é usar os dados do passado dia a dia. Então, para a mesma porta e mesmo dia de previsão, montamos uma tabela como esta:

| dia | n_CA | n_FC | ... | n_total |
|---|---:|---:|---:|---:|
| 17/05 | 0 | 1 | ... | 1 |
| 18/05 | 0 | 0 | ... | 0 |
| 19/05 | 2 | 0 | ... | 2 |
| ... | ... | ... | ... | ... |
| 14/06 | 2 | 0 | ... | 2 |

Essa tabela tem:

- 30 linhas, uma para cada dia anterior ao t0
- 12 colunas, uma para cada feature daquele dia

Ou seja, uma amostra da RNN é:

- sequência de 30 dias
- cada dia com 12 features
- organizada na ordem em que os eventos aconteceram

A data de previsão t0 não entra na sequência. A sequência termina no dia anterior, `14/06`, para evitar vazamento de informação do futuro.

Assim, a entrada da RNN tem formato:

- [n_amostras, 30, 12]

Onde:

- 30 = quantidade de dias na sequência
- 12 = quantidade de features por dia

Por isso, o treino da LSTM não recebe uma linha com janelas móveis. Ele recebe o histórico dos últimos 30 dias em sequência, e é a própria rede que aprende quais padrões do tempo importam para prever a falha.

Em resumo:

- Random Forest: uma linha por porta por dia, com features já calculadas
- RNN: uma sequência de 30 dias por porta, com as features diárias em ordem

Essa diferença é justamente o que faz a RNN aprender padrões temporais, em vez de depender só de agregados pré-definidos.


In [1]:
import os
# a instalacao do torch neste ambiente usa uma OpenMP diferente da do numpy/scikit-learn
# (MKL); sem isso, e sem limitar a 1 thread, o treino trava (nao da erro, so nunca avanca)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import numpy as np
import pandas as pd
import torch
torch.set_num_threads(1)
import torch.nn as nn
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              average_precision_score, roc_auc_score, confusion_matrix)

torch.manual_seed(42)
np.random.seed(42)

d = np.load('sequencias.npz')
X, y = d['X'], d['y']
idx = pd.read_parquet('sequencias_index.parquet')
print(f'X: {X.shape} | y: {y.shape} | taxa de positivos: {y.mean()*100:.2f}%')

X: (306864, 30, 12) | y: (306864,) | taxa de positivos: 8.09%


## 5. Separação treino/teste

Foi feito o mesmo corte temporal do Modelo 5. Com treino até 23/04/2025, teste a partir de 01/05/2025 (7 dias de folga entre os dois), e uma parte de validação dentro do treino (a partir de 31/01/2025), que foi usada aqui pra decidir quantas épocas treinar e qual limiar de decisão usar.

In [2]:
FIM_TREINO, INI_TESTE, INI_VALID = (pd.Timestamp('2025-04-23'), pd.Timestamp('2025-05-01'),
                                     pd.Timestamp('2025-01-31'))
data = idx['Data']
m_treino = data <= FIM_TREINO
m_teste = data >= INI_TESTE
m_fit = m_treino & (data <= INI_VALID - pd.Timedelta(days=8))
m_val = m_treino & (data >= INI_VALID)

print(f'fit: {m_fit.sum():,} | val: {m_val.sum():,} | treino: {m_treino.sum():,} | teste: {m_teste.sum():,}')
print(f'taxa falha fit: {y[m_fit.values].mean()*100:.2f}% | val: {y[m_val.values].mean()*100:.2f}% | teste: {y[m_teste.values].mean()*100:.2f}%')

fit: 194,304 | val: 35,520 | treino: 232,992 | teste: 71,184
taxa falha fit: 7.64% | val: 9.28% | teste: 8.71%


## 6. Normalização

Diferente do Random Forest que não precisa de normalização, a RNN aprende melhor quando os números de entrada estão numa escala parecida. Sem isso, uma feature com valores grandes (tipo `dur_max`, em segundos) tem um peso maior no treino só por causa da escala, não porque é mais importante.

In [3]:
mu = X[m_fit.values].mean(axis=(0, 1), keepdims=True)
sd = X[m_fit.values].std(axis=(0, 1), keepdims=True) + 1e-6


def norm(a):
    return (a - mu) / sd


device = torch.device('cpu')  # o LSTM no MPS (GPU do Apple Silicon) e mal suportado nessa versao do torch

## 7. A arquitetura escolhida

- **LSTM(12, 32)**: recebe as 12 features de cada dia e mantém um estado escondido de 32 números ao longo da sequência. 32 é pequeno por ser a primeira tentativa, sem uma busca de hiperparâmetro ainda.
- **Uma camada**: dá pra juntar várias LSTMs para capturar padrões mais complexos, mas isso também exige mais dado pra não decorar o treino. Começar com uma camada segue o mesmo raciocínio do `max_depth` limitado no Random Forest: começar simples e complicar se presicar depois.
- **Camada linear de saída**: pega o estado escondido do último dia da sequência (32 números) e transforma num único número (chamado de *logit*), que depois vira uma probabilidade entre 0 e 1 aplicando a função sigmoid.
- **pos_weight na função de perda**: dá peso maior aos exemplos positivos (porta que quebrou) no cálculo do erro, porque só ~8% das linhas são positivas.

In [4]:
class RNNFalha(nn.Module):
    def __init__(self, n_feat, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, batch_first=True)
        self.saida = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)      # h: estado escondido do ultimo dia da sequencia
        return self.saida(h[-1]).squeeze(-1)


def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)


def treina(Xtr, ytr, Xval, yval, max_epocas=15, paciencia=3):
    modelo = RNNFalha(Xtr.shape[-1]).to(device)
    pos_weight = torch.tensor([float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0)],
                               dtype=torch.float32, device=device)
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    otim = torch.optim.Adam(modelo.parameters(), lr=1e-3)

    Xtr_t, ytr_t = to_tensor(norm(Xtr)), to_tensor(ytr)
    Xval_t = to_tensor(norm(Xval)).to(device)
    n = len(Xtr_t)
    lote = 1024

    melhor_prauc, melhor_epoca, sem_melhora = -1, 0, 0
    for epoca in range(1, max_epocas + 1):
        modelo.train()
        perm = torch.randperm(n)
        perda_total = 0.0
        for i in range(0, n, lote):
            b = perm[i:i + lote]
            xb, yb = Xtr_t[b].to(device), ytr_t[b].to(device)
            otim.zero_grad()
            logit = modelo(xb)
            perda = perda_fn(logit, yb)
            perda.backward()
            otim.step()
            perda_total += perda.item() * len(b)

        modelo.eval()
        with torch.no_grad():
            prob_val = torch.sigmoid(modelo(Xval_t)).cpu().numpy()
        prauc = average_precision_score(yval, prob_val)
        print(f'epoca {epoca:2d} | perda={perda_total/n:.4f} | PR-AUC val={prauc:.4f}')

        if prauc > melhor_prauc:
            melhor_prauc, melhor_epoca, sem_melhora = prauc, epoca, 0
        else:
            sem_melhora += 1
            if sem_melhora >= paciencia:
                print(f'parando cedo na epoca {epoca} (melhor foi {melhor_epoca})')
                break
    return melhor_epoca, melhor_prauc

## 8. Épocas e perda

Diferente do Random Forest, que treina uma vez (monta as árvores e acabou), a rede neural aprende aos poucos; cada **época** é uma passada completa pelos dados de treino, ajustando um pouco os pesos da rede a cada passada. A perda (*loss*) mede o quão erradas estão as previsões naquela época.

Mas treinar por mais épocas nem sempre é melhor: depois de um ponto, a rede começa a decorar o treino em vez de aprender o padrão geral (*overfitting*), e passa a ir pior em dados novos. Por isso o número de épocas é escolhido observando o PR-AUC na validação. O modelo para de treinar quando a validação não melhora por 3 épocas seguidas (paciência), e fica com a melhor época encontrada.

In [5]:
melhor_epoca, melhor_prauc = treina(X[m_fit.values], y[m_fit.values], X[m_val.values], y[m_val.values])
print(f'\nmelhor epoca: {melhor_epoca} | PR-AUC val: {melhor_prauc:.4f}')

epoca  1 | perda=1.2453 | PR-AUC val=0.1537
epoca  2 | perda=1.2095 | PR-AUC val=0.1590
epoca  3 | perda=1.2009 | PR-AUC val=0.1621
epoca  4 | perda=1.1963 | PR-AUC val=0.1675
epoca  5 | perda=1.1898 | PR-AUC val=0.1657
epoca  6 | perda=1.1879 | PR-AUC val=0.1652
epoca  7 | perda=1.1845 | PR-AUC val=0.1663
parando cedo na epoca 7 (melhor foi 4)

melhor epoca: 4 | PR-AUC val: 0.1675


A perda cai de forma consistente a cada época, mas o PR-AUC de validação para de melhorar depois da época 4 e passa a oscilar. Ou seja sinal de que a rede já aprendeu o que dava pra aprender da sequência com essa arquitetura simples, e treinar mais não ajudaria. Por isso o treino parou na época 7, guardando o modelo da época 4.

## 9. Treino final e avaliação no teste

Aqui foi utilizada a mesma lógica do Modelo 5: o primeiro o modelo é treinado no tr_fit, e o limiar de decisão é escolhido no tr_val maximizando F1. Depois o modelo é retreinado do zero no treino completo (`tr_fit` + `tr_val`), pelas mesmas 4 épocas encontradas no passo anterior, e avaliado uma vez no teste. O teste não é usado pra decidir nada, só pra medir o resultado final.

In [6]:
def melhor_limiar(y, prob):
    grid = np.unique(np.quantile(prob, np.linspace(0.50, 0.999, 200)))
    return max(((f1_score(y, (prob >= t).astype(int), zero_division=0), t) for t in grid))[1]


def avalia(nome, yte, pred, prob):
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    return {'modelo': nome, 'precisao': precision_score(yte, pred, zero_division=0),
            'recall': recall_score(yte, pred, zero_division=0), 'f1': f1_score(yte, pred, zero_division=0),
            'pr_auc': average_precision_score(yte, prob), 'roc_auc': roc_auc_score(yte, prob),
            'alertas': int(pred.sum()), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn)}


def treina_fixo(Xtr, ytr, n_epocas):
    """Treina por um numero fixo de epocas, sem early stopping (usado no refit final)."""
    modelo = RNNFalha(Xtr.shape[-1]).to(device)
    pos_weight = torch.tensor([float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0)],
                               dtype=torch.float32, device=device)
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    otim = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    Xtr_t, ytr_t = to_tensor(norm(Xtr)), to_tensor(ytr)
    n = len(Xtr_t)
    lote = 1024
    for epoca in range(1, n_epocas + 1):
        modelo.train()
        perm = torch.randperm(n)
        for i in range(0, n, lote):
            b = perm[i:i + lote]
            xb, yb = Xtr_t[b].to(device), ytr_t[b].to(device)
            otim.zero_grad()
            perda = perda_fn(modelo(xb), yb)
            perda.backward()
            otim.step()
    return modelo


def prediz(modelo, Xa):
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(modelo(to_tensor(norm(Xa)).to(device))).cpu().numpy()


modelo_fit = treina_fixo(X[m_fit.values], y[m_fit.values], melhor_epoca)
prob_val = prediz(modelo_fit, X[m_val.values])
limiar = melhor_limiar(y[m_val.values], prob_val)
print(f'limiar escolhido (mesma logica do Modelo 5): {limiar:.4f}')

modelo_final = treina_fixo(X[m_treino.values], y[m_treino.values], melhor_epoca)
prob_teste = prediz(modelo_final, X[m_teste.values])
pred_teste = (prob_teste >= limiar).astype(int)
resultado_rnn = avalia('RNN (LSTM)', y[m_teste.values], pred_teste, prob_teste)
print(pd.DataFrame([resultado_rnn]).round(3).to_string(index=False))

limiar escolhido (mesma logica do Modelo 5): 0.5772
    modelo  precisao  recall    f1  pr_auc  roc_auc  alertas   tp    fp   fn
RNN (LSTM)     0.137   0.487 0.214   0.154    0.661    22038 3020 19018 3182


## 10. Comparação com o Modelo 5

A tabela usa o resultado do Random Forest com features selecionadas (`linha_final` do `modelo_5.ipynb`), que é o modelo usado na prática pelo Modelo 5. Ele foi avaliado com o mesmo alvo, a mesma partição temporal e a mesma métrica que a RNN. Assim, a diferença observada nos resultados pode ser atribuída à arquitetura, e não a uma mudança no que está sendo previsto.

In [7]:
# valores do Modelo 5 (modelo_5.ipynb, secao 6 -- "Random Forest (features selecionadas)")
resultado_rf = {'modelo': 'Random Forest (features selecionadas)', 'precisao': 0.159, 'recall': 0.553,
                 'f1': 0.247, 'pr_auc': 0.185, 'roc_auc': 0.697, 'alertas': 21564, 'tp': 3432, 'fp': 18132, 'fn': 2770}

R = pd.DataFrame([resultado_rf, resultado_rnn])
print(R.round(3).to_string(index=False))

                               modelo  precisao  recall    f1  pr_auc  roc_auc  alertas   tp    fp   fn
Random Forest (features selecionadas)     0.159   0.553 0.247   0.185    0.697    21564 3432 18132 2770
                           RNN (LSTM)     0.137   0.487 0.214   0.154    0.661    22038 3020 19018 3182


| Modelo | Precisão | Recall | F1 | PR-AUC | ROC-AUC |
|---|---|---|---|---|---|
| Random Forest (Modelo 5) | 15,9% | 55,3% | 0,247 | 0,185 | 0,697 |
| RNN (LSTM) | 13,7% | 48,7% | 0,214 | 0,154 | 0,661 |

Usar PR-AUC pra comparar aqui não contradiz o recall ser a prioridade de negócio, pois são duas perguntas diferentes:

- **Recall** responde: "com o limiar de decisão que a gente escolheu, quantas falhas reais o modelo realmente pega?" É a métrica que importa na hora de operar o modelo, porque deixar passar uma falha é pior do que mandar checar uma porta à toa.
- **PR-AUC** responde: "no geral, sem fixar nenhum limiar, o modelo separa bem os casos de risco dos de não-risco?" É a métrica pra comparar duas arquiteturas diferentes (Random Forest x RNN), porque cada uma pode ter escolhido um limiar diferente ,comparar só o recall direto seria injusto nesse caso.

Resumindo: recall continua sendo o que decide se o modelo está pronto pra operação; PR-AUC é só o que usamos aqui pra saber qual arquitetura aprendeu melhor o padrão, antes de chegar num limiar específico.

Nesta primeira tentativa, a RNN ficou abaixo do Random Forest em todas as métricas com recall 48,7% contra 55,3%, precisão 13,7% contra 15,9%, PR-AUC 0,154 contra 0,185. Entretanto, isso não invalida a arquitetura recorrente, a comparação usa uma LSTM simples (uma camada, 32 unidades, sem busca de hiperparâmetros), enquanto o Random Forest já passou por seleção de features via SHAP e por RandomizedSearchCV`/`GridSearchCV.

Também é esperado que o Random Forest fosse melhor. Isso porque as janelas móveis (1/3/7/14/30 dias) já são um resumo desenhado à mão pra esse problema específico, por isso que funcionaram bem no Modelo 5. A RNN precisaria redescobrir essas agregações sozinha, partindo de menos informação por linha (12 features cruas por dia contra 71 já elaboradas) e sem nenhum ajuste de hiperparâmetros.

## 11. Síntese

A RNN aprendeu sinal real da sequência de alarmes, PR-AUC de 0,154 fica bem acima da taxa de falha do teste (8,71%), que seria a referência de um modelo sem poder preditivo nenhum. Mas nesta primeira versão não superou o Random Forest do Modelo 5.

Próximos passos possíveis: aumentar a rede (mais unidades ou camadas), testar GRU, ou rodar uma busca de hiperparâmetros equivalente à que foi feita pro Random Forest antes de comparar as arquiteturas de forma definitiva.